## Creating Evaluation Data for the AI4SNOW Project

### Author: Celia Baumhoer (2025)

In [2]:
!micromamba create -y -n myenv jupyter xarray rioxarray odc-stac odc-geo pystac-client dask folium geopandas zarr jupyter-server-proxy

/bin/bash: micromamba: command not found


In [3]:
!pip install rasterio

import os
from glob import glob

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely.geometry import Point, box

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Save patches with unifrom FSC distribution in bins


We want to have 100 samples per FSC bin. Bins are definde as 0, >1 to 10, >10 to 20,....,100 FSC.

In [4]:
# test data Canada
# resample_dir = "../Landsat/fsc_test_rockies_100m"
# train data globally
resample_dir = "../Landsat/fsc_train_100m"
patch_dir = "../Landsat/fsc_test_june"

# use RGI 7.0 to exclude glaciers world wide
# https://www.glims.org/rgi_user_guide/02_regions_definition.html
# Load all RGI glacier shapefiles
rgi_shapefiles = glob("../aux_data/RGI/*.shp")

In [5]:
# Create output folder
os.makedirs(patch_dir, exist_ok=True)

# Define FSC bins explicitly
bin_edges = [0, 1, 11, 21, 31, 41, 51, 61, 71, 81, 91, 100]
# bin_edges = [11, 21, 31, 41, 51, 61, 71, 81, 91]
bin_labels = [(bin_edges[i], bin_edges[i + 1] - 1) for i in range(1, len(bin_edges) - 1)]
print(bin_edges)
bin_labels = [(0, 0)] + bin_labels + [(100, 100)]
print(bin_labels)
# bin_labels = [(100, 100)]
# Resulting bins:
# (0, 0), (1, 10), (11, 20), (21, 30), ..., (91, 99), (100, 100)

# Number of patches to sample per bin
# patches_per_bin = 1
# patch_size = 50
patches_per_bin = 100
patch_size = 10

[0, 1, 11, 21, 31, 41, 51, 61, 71, 81, 91, 100]
[(0, 0), (1, 10), (11, 20), (21, 30), (31, 40), (41, 50), (51, 60), (61, 70), (71, 80), (81, 90), (91, 99), (100, 100)]


In [6]:
import os

num_files = sum(1 for f in os.listdir(patch_dir) if os.path.isfile(os.path.join(patch_dir, f)))

print(f"Number of files: {num_files}")

Number of files: 0


In [7]:
for raster_path in glob(os.path.join(resample_dir, "*.tif")):
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        height, width = data.shape
        transform = src.transform
        crs = src.crs
        nodata = src.nodata
        bounds = src.bounds
        raster_box = box(*bounds)  # raster extent as shapely box

        # Find glaciers overlapping this raster
        overlapping_glaciers = []
        for shp_path in rgi_shapefiles:
            gdf = gpd.read_file(shp_path)
            if gdf.crs != crs:
                gdf = gdf.to_crs(crs)
            gdf = gdf[gdf.intersects(raster_box)]
            if not gdf.empty:
                overlapping_glaciers.append(gdf)

        # Merge all overlapping glaciers into a single GeoDataFrame
        if overlapping_glaciers:
            glacier_gdf = gpd.GeoDataFrame(
                pd.concat(overlapping_glaciers, ignore_index=True), crs=crs
            )
            glacier_union = glacier_gdf.union_all()  # single polygon to exclude
        else:
            glacier_union = None  # no glaciers to exclude

        # Track how many patches we have per bin
        bin_counts = {bin_label: 0 for bin_label in bin_labels}

        max_attempts = 500000
        attempts = 0

        while min(bin_counts.values()) < patches_per_bin and attempts < max_attempts:
            attempts += 1

            # Generate a random point
            x = np.random.uniform(
                bounds.left + patch_size * transform.a, bounds.right - patch_size * transform.a
            )
            y = np.random.uniform(
                bounds.bottom + patch_size * abs(transform.e),
                bounds.top - patch_size * abs(transform.e),
            )
            point = Point(x, y)

            # Skip if point is inside a glacier
            if glacier_union and glacier_union.contains(point):
                continue

            col, row = ~transform * (x, y)
            row, col = int(row), int(col)

            # Skip if patch would go out of bounds
            if row < patch_size // 2 or row >= height - patch_size // 2:
                continue
            if col < patch_size // 2 or col >= width - patch_size // 2:
                continue

            window = Window(col - patch_size // 2, row - patch_size // 2, patch_size, patch_size)
            patch = src.read(1, window=window)

            if np.isnan(patch).any():
                continue

            patch_mean = np.mean(patch) * 100  # FSC scale 0-100

            # Find which bin this patch belongs to
            # for bin_min, bin_max in bin_labels:
            #    if (bin_min == 0 and patch_mean == 0) or (patch_mean > bin_min and patch_mean <= bin_max):
            #        bin_label = (bin_min, bin_max)
            #        break

            for bin_min, bin_max in bin_labels:
                if bin_min <= patch_mean <= bin_max:
                    bin_label = (bin_min, bin_max)
                    break
            else:
                continue  # patch doesn't fit any bin

            if bin_counts[bin_label] >= patches_per_bin:
                continue

            patch_transform = src.window_transform(window)
            scene_name = os.path.basename(raster_path)[:4]
            date_str = "00000000"
            parts = os.path.basename(raster_path).split("_")
            for part in parts:
                if len(part) == 8 and part.isdigit():
                    date_str = part
                    break

            filename = f"{scene_name}_{date_str}_FSC{int(patch_mean)}_{y:.5f}_{x:.5f}.tif"
            patch_path = os.path.join(patch_dir, filename)

            patch_profile = src.profile.copy()
            patch_profile.update(
                {
                    "height": patch_size,
                    "width": patch_size,
                    "transform": patch_transform,
                    "compress": "deflate",
                    "dtype": "float32",
                    "nodata": nodata,
                    "crs": crs,
                }
            )

            with rasterio.open(patch_path, "w", **patch_profile) as dst:
                dst.write(patch.astype(np.float32), 1)

            bin_counts[bin_label] += 1

        print(f"Finished sampling for {os.path.basename(raster_path)}: {bin_counts}")

Finished sampling for LC09_L2SP_147038_20220606_20220608_02_T1_snowclass_QA.tif: {(0, 0): 100, (1, 10): 100, (11, 20): 100, (21, 30): 100, (31, 40): 100, (41, 50): 100, (51, 60): 100, (61, 70): 100, (71, 80): 100, (81, 90): 100, (91, 99): 100, (100, 100): 100}
Finished sampling for LC09_L2SP_140041_20230216_20230310_02_T1_snowclass_QA.tif: {(0, 0): 100, (1, 10): 100, (11, 20): 100, (21, 30): 100, (31, 40): 100, (41, 50): 100, (51, 60): 100, (61, 70): 100, (71, 80): 100, (81, 90): 100, (91, 99): 100, (100, 100): 100}
Finished sampling for LC09_L2SP_136040_20230119_20230313_02_T1_snowclass_QA.tif: {(0, 0): 100, (1, 10): 100, (11, 20): 100, (21, 30): 100, (31, 40): 100, (41, 50): 100, (51, 60): 100, (61, 70): 100, (71, 80): 100, (81, 90): 100, (91, 99): 100, (100, 100): 100}
Finished sampling for LC09_L2SP_152035_20230119_20230313_02_T1_snowclass_QA.tif: {(0, 0): 100, (1, 10): 100, (11, 20): 100, (21, 30): 100, (31, 40): 100, (41, 50): 100, (51, 60): 100, (61, 70): 100, (71, 80): 100, (81

In [9]:
from pathlib import Path

# navigate into the masks folder
folder_path = Path("../Landsat/fsc_test_june/")

file_count = sum(1 for item in folder_path.iterdir() if item.is_file())

print(file_count)

36160


## Special Case Switzerland

We need to make sure the patches are actually located in Switzerland as the scene boundaries of the scenes are larger than Switzerland itself. Furthermore, we use a more accurate glacier boundary from Paul et al. 2020.

In [47]:
import geopandas as gpd

switzerland_gdf = gpd.read_file(
    "/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/aux_data/Switzerland.gpkg"
)
switzerland_polygon = switzerland_gdf.union_all()  # combine all geometries into one

# not over glaciated area
# this shp is more accurate than the worldwide RGI from year 2000
# https://essd.copernicus.org/articles/12/1805/2020/essd-12-1805-2020.html
glacier_shp_path = (
    "/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/aux_data/alpine_glaciers_paul.gpkg"
)
glacier_gdf = gpd.read_file(glacier_shp_path)
glacier_gdf = glacier_gdf.to_crs("EPSG:4326")  # adjust CRS if needed
glacier_polygon = glacier_gdf.union_all()

In [48]:
resample_dir = (
    "/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/Landsat/fsc_test_switzerland_100m"
)
patch_dir = f"{resample_dir}_patches"

# Output folder
os.makedirs(patch_dir, exist_ok=True)

# FSC bins
bin_labels = [
    (0, 0),
    (1, 10),
    (11, 20),
    (21, 30),
    (31, 40),
    (41, 50),
    (51, 60),
    (61, 70),
    (71, 80),
    (81, 90),
    (91, 99),
    (100, 100),
]

patches_per_bin = 10
patch_size = 10

In [49]:
for raster_path in glob(os.path.join(resample_dir, "*.tif")):
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        height, width = data.shape
        transform = src.transform
        crs = src.crs
        nodata = src.nodata
        bounds = src.bounds

        # Transform polygons to raster CRS
        switzerland_polygon_raster_crs = (
            gpd.GeoSeries([switzerland_polygon], crs="EPSG:4326").to_crs(crs).iloc[0]
        )
        glacier_polygon_raster_crs = (
            gpd.GeoSeries([glacier_polygon], crs="EPSG:4326").to_crs(crs).iloc[0]
        )

        # Track how many patches we have per bin
        bin_counts = {bin_label: 0 for bin_label in bin_labels}

        max_attempts = 500000
        attempts = 0

        while min(bin_counts.values()) < patches_per_bin and attempts < max_attempts:
            attempts += 1

            # Generate a random point
            x = np.random.uniform(
                bounds.left + patch_size * transform.a, bounds.right - patch_size * transform.a
            )
            y = np.random.uniform(
                bounds.bottom + patch_size * abs(transform.e),
                bounds.top - patch_size * abs(transform.e),
            )
            point = Point(x, y)

            # Check if point is inside Switzerland and not in glacier
            if not switzerland_polygon_raster_crs.contains(point):
                continue
            if glacier_polygon_raster_crs.contains(point):
                continue

            col, row = ~transform * (x, y)
            row, col = int(row), int(col)

            # Skip if patch would go out of raster bounds
            if row < patch_size // 2 or row >= height - patch_size // 2:
                continue
            if col < patch_size // 2 or col >= width - patch_size // 2:
                continue

            window = Window(col - patch_size // 2, row - patch_size // 2, patch_size, patch_size)
            patch = src.read(1, window=window)

            if np.isnan(patch).any():
                continue

            patch_mean = np.mean(patch) * 100

            # Find the bin for this patch
            for bin_min, bin_max in bin_labels:
                if bin_min <= patch_mean <= bin_max:
                    bin_label = (bin_min, bin_max)
                    break
            else:
                continue

            if bin_counts[bin_label] < patches_per_bin:
                patch_transform = src.window_transform(window)

                scene_name = os.path.basename(raster_path)[:4]
                date_str = "00000000"
                try:
                    parts = os.path.basename(raster_path).split("_")
                    for part in parts:
                        if len(part) == 8 and part.isdigit():
                            date_str = part
                            break
                except:
                    pass

                filename = f"{scene_name}_{date_str}_FSC{int(patch_mean)}_{y:.5f}_{x:.5f}.tif"
                patch_path = os.path.join(patch_dir, filename)

                patch_profile = src.profile.copy()
                patch_profile.update(
                    {
                        "height": patch_size,
                        "width": patch_size,
                        "transform": patch_transform,
                        "compress": "deflate",
                        "dtype": "float32",
                        "nodata": nodata,
                        "crs": crs,
                    }
                )

                with rasterio.open(patch_path, "w", **patch_profile) as dst:
                    dst.write(patch.astype(np.float32), 1)

                bin_counts[bin_label] += 1

        print(f"Finished sampling {os.path.basename(raster_path)}: {bin_counts}")

Finished sampling LC08_L2SP_195028_20201030_20201106_02_T1_snowclass_QA.tif: {(0, 0): 10, (1, 10): 10, (11, 20): 10, (21, 30): 10, (31, 40): 10, (41, 50): 10, (51, 60): 10, (61, 70): 10, (71, 80): 10, (81, 90): 10, (91, 99): 10, (100, 100): 10}
Finished sampling LC08_L2SP_194028_20201226_20210310_02_T1_snowclass_QA.tif: {(0, 0): 10, (1, 10): 10, (11, 20): 10, (21, 30): 10, (31, 40): 10, (41, 50): 10, (51, 60): 10, (61, 70): 10, (71, 80): 10, (81, 90): 10, (91, 99): 10, (100, 100): 10}
Finished sampling LC08_L2SP_195028_20210118_20210306_02_T1_snowclass_QA.tif: {(0, 0): 10, (1, 10): 10, (11, 20): 10, (21, 30): 10, (31, 40): 10, (41, 50): 10, (51, 60): 10, (61, 70): 10, (71, 80): 10, (81, 90): 10, (91, 99): 10, (100, 100): 10}
Finished sampling LC08_L2SP_196028_20200903_20200918_02_T1_snowclass_QA.tif: {(0, 0): 10, (1, 10): 10, (11, 20): 10, (21, 30): 7, (31, 40): 9, (41, 50): 2, (51, 60): 0, (61, 70): 2, (71, 80): 2, (81, 90): 0, (91, 99): 5, (100, 100): 0}
Finished sampling LC08_L2SP_1

## Sanity check map of distribution

In [64]:
import os

import folium
import rasterio
from pyproj import Transformer
from rasterio.windows import Window

patch_points = []

for patch_file in os.listdir(patch_dir):
    if patch_file.endswith(".tif"):
        patch_path = os.path.join(patch_dir, patch_file)
        with rasterio.open(patch_path) as src:
            # Center of the patch in raster coordinates
            center_col = src.width / 2
            center_row = src.height / 2

            # Convert to spatial coordinates
            center_x, center_y = src.transform * (center_col, center_row)

            # Transform to WGS84
            transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)
            lon, lat = transformer.transform(center_x, center_y)

            # Extract FSC from filename
            import re

            match = re.search(r"FSC(\d+)_", patch_file)
            fsc = int(match.group(1)) if match else None

            patch_points.append({"fsc": fsc, "lat": lat, "lon": lon})

# Create Folium map centered in Switzerland
# m = folium.Map(location=[46.8, 8.3], zoom_start=7)
# Create Folium map centered in Canada
# m = folium.Map(location=[57, -130], zoom_start=4)
# Create Folium map centered Northern Hemisphere
m = folium.Map(location=[48, -11], zoom_start=2)


# Function to color patches by FSC
def fsc_color(fsc):
    """
    Returns a color from gray (0) to light blue (low FSC) to dark blue (high FSC, 100).
    """
    if fsc == 0:
        return "gray"
    else:
        # Normalize FSC to 1-100
        fsc_norm = min(max(fsc, 1), 100)
        # Generate a blue gradient: light blue for low FSC, dark blue for high FSC
        # Using hex colors from #ADD8E6 (light blue) to #00008B (dark blue)
        # Interpolate RGB components
        light_rgb = np.array([173, 216, 230])  # #ADD8E6
        dark_rgb = np.array([0, 0, 139])  # #00008B
        ratio = (fsc_norm - 1) / 99  # normalized between 0 and 1
        rgb = (light_rgb * (1 - ratio) + dark_rgb * ratio).astype(int)
        return f"#{rgb[0]:02x}{rgb[1]:02x}{rgb[2]:02x}"


# Add patch centers as CircleMarkers
for pt in patch_points:
    folium.CircleMarker(
        location=[pt["lat"], pt["lon"]],
        radius=3,
        color=fsc_color(pt["fsc"]),
        fill=True,
        fill_opacity=0.7,
        popup=f"FSC: {pt['fsc']}",
    ).add_to(m)

# Save map
# grey FSC 0 to dark blue FSC 100
m
# m.save("/dss/dsstbyfs02/pn49ci/pn49ci-dss-0013/evaluation/figures/train_patches_global.html")
# print("Map saved to patches_map.html")